# 05_Factor_Stability — 因子穩定性年度分析

## 目的

Phase 3-A 核心研究：找出在**所有年份都穩定有效**的因子。

**背景**：在 Phase 2 的 Fold-Internal IC 分析中，發現以下因子穩定性差異：

| 因子 | Fold 選中次數 (out of 24) | 狀態 |
|------|--------------------------|------|
| `vol_ratio` | 24/24 | 非常穩定 |
| `mom_1m_ra` | 23/24 | 非常穩定 |
| `trust_net_10d` | 22/24 | 穩定 |
| `mom_1m` | 21/24 | 穩定 |
| `price_52w` | **8/24** | **不穩定** |
| `foreign_net_20d` | **3/24** | **幾乎無效** |

本 Notebook 進一步按**年份**分解每個因子的 ICIR，找出：
- 方向每年一致的因子 → 可信賴的 alpha 信號
- 某年正/某年負的因子 → 噪音，不應依賴

## 設計原理（Single Source of Truth）

為了確保資料清理、過濾、流動性處理、因子計算 **完全等同於 strategy.py**，我們直接動態執行 `strategy.py` 的前半部，攔截其產生的 `X` 和 `y` 變數。不重複造輪子，保證 100% 同步。

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import spearmanr
from tqdm import tqdm

print("Executing data generation pipeline from strategy.py (takes ~20 seconds)...")

# 儲存原本的工作路徑
original_cwd = os.getcwd()

try:
    # 將執行路徑強制切換回專案根目錄，這樣 strategy.py 裡面的 finmind_cache/ 相對路徑才找得到東西！
    if os.path.basename(original_cwd) == "Research":
        os.chdir("..")
        
    with open("strategy.py", "r", encoding="utf-8") as f:
        code = f.read()

    # 找到 Walk-Forward 的註解，切斷後面的模型訓練與回測邏輯
    cutoff = code.find("Walk-Forward Purged training")
    cutoff_line = code.rfind("\n", 0, cutoff)
    exe_code = code[:cutoff_line]

    env = {}
    exec(exe_code, env)

    X = env["X"]
    y = env["y"]
    test_dates = [d for d in env["all_dates"] if d >= env["TEST_START"]]
    print(f"\nPipeline finished! Extracted X shape: {X.shape}, test_dates: {len(test_dates)}")
finally:
    # 確保不管執行成不成功，都切換回原本的 Research 資料夾
    os.chdir(original_cwd)


Executing data generation pipeline from strategy.py (takes ~20 seconds)...
  Taiwan ML Stock Strategy  v4  (Optimized)

[1/7] Loading data...


FileNotFoundError: [Errno 2] No such file or directory: 'finmind_cache/close_wide.pkl'

In [ ]:
# ── 按年計算每個因子的月度 IC，整理成熱力圖格式
years = sorted(set(d.year for d in test_dates))
ic_by_year = {}

for factor in tqdm(X.columns, desc="Computing IC by year"):
    ic_by_year[factor] = {yr: [] for yr in years}
    for d in test_dates:
        try:
            if d in X.index.get_level_values(0):
                xf = X.xs(d, level=0)[factor].dropna()
                yf = y.xs(d, level=0).reindex(xf.index).dropna()
                xf = xf.reindex(yf.index)
                if len(yf) > 10:
                    ic_val, _ = spearmanr(xf.values, yf.values)
                    if not np.isnan(ic_val):
                        ic_by_year[factor][d.year].append(ic_val)
        except Exception:
            pass

print("IC computation done")

In [ ]:
# ── 彙整成 ICIR 熱力圖資料
icir_matrix = {}

for factor in X.columns:
    icir_matrix[factor] = {}
    for yr in years:
        ics = pd.Series(ic_by_year[factor][yr])
        if len(ics) >= 3:
            icir = ics.mean() / (ics.std() + 1e-8)
        else:
            icir = np.nan
        icir_matrix[factor][yr] = icir

icir_df = pd.DataFrame(icir_matrix).T

# 排序：依全期平均 |ICIR| 降序
overall_abs_icir = icir_df.abs().mean(axis=1).sort_values(ascending=False)
icir_df = icir_df.loc[overall_abs_icir.index]

print("ICIR Heatmap data (row=factor, col=year):")
display(icir_df.round(3))

In [ ]:
# ── 視覺化：ICIR 年度熱力圖
fig = go.Figure(go.Heatmap(
    z           = icir_df.values,
    x           = [str(y) for y in icir_df.columns],
    y           = icir_df.index.tolist(),
    colorscale  = "RdBu",
    zmid        = 0,
    text        = icir_df.round(2).values,
    texttemplate= "%{text}",
    showscale   = True,
    colorbar    = dict(title="ICIR"),
))
fig.update_layout(
    title      = "Factor ICIR by Year (2020~2026) — Red=Bearish factor, Blue=Bullish factor",
    xaxis_title= "Year",
    yaxis_title= "Factor",
    template   = "plotly_dark",
    height     = 550,
)
fig.show()
print("閱讀指引：藍色=該年因子有效（負值因子用於空頭），紅色=反向，白色=無效")
print("理想的因子：每年都是同色（方向一致），不出現顏色在年份間切換")

In [ ]:
# ── 因子穩定性評分：方向一致性（sign consistency）
stability_report = []
for factor in icir_df.index:
    vals = icir_df.loc[factor].dropna()
    if len(vals) == 0: continue
    pct_negative = (vals < 0).mean()
    pct_positive = (vals > 0).mean()
    sign_consistency = max(pct_negative, pct_positive)
    avg_abs_icir     = vals.abs().mean()
    direction        = "NEGATIVE (contrarian)" if pct_negative > pct_positive else "POSITIVE (momentum)"
    stability_report.append({
        "Factor"           : factor,
        "Direction"        : direction,
        "Sign Consistency" : f"{sign_consistency:.0%}",
        "Avg |ICIR|"       : f"{avg_abs_icir:.3f}",
        "Recommendation"   : "KEEP" if sign_consistency >= 0.80 else ("REVIEW" if sign_consistency >= 0.60 else "DROP"),
    })

stability_df = pd.DataFrame(stability_report).set_index("Factor")
print("Factor Stability Report:")
display(stability_df)

In [ ]:
# ── ICIR 逐年折線圖（每個因子一條線）
keep_factors   = stability_df[stability_df["Recommendation"] == "KEEP"].index.tolist()
review_factors = stability_df[stability_df["Recommendation"] == "REVIEW"].index.tolist()
drop_factors   = stability_df[stability_df["Recommendation"] == "DROP"].index.tolist()

fig2 = go.Figure()
year_labels = [str(y) for y in icir_df.columns]

for factor in icir_df.index:
    rec = stability_df.loc[factor, "Recommendation"]
    color = {"KEEP": "#2962FF", "REVIEW": "#FFA000", "DROP": "#E53935"}[rec]
    dash  = {"KEEP": "solid", "REVIEW": "dash", "DROP": "dot"}[rec]
    fig2.add_trace(go.Scatter(
        x=year_labels, y=icir_df.loc[factor].values,
        name=f"{factor} ({rec})", mode="lines+markers",
        line=dict(color=color, dash=dash, width=2),
    ))

fig2.add_hline(y=0, line_color="white", line_dash="dash", line_width=1)
fig2.update_layout(
    title   = "Factor ICIR Trend by Year — Blue=KEEP, Orange=REVIEW, Red=DROP",
    xaxis_title="Year", yaxis_title="ICIR",
    template="plotly_dark", height=500,
    legend=dict(x=1.01, y=0.99),
)
fig2.show()

In [ ]:
# ── 最終建議
print("="*60)
print("PHASE 3-A 結論：因子篩選建議")
print("="*60)
print(f"\n  KEEP  ({len(keep_factors)} 個）：{keep_factors}")
print(f"  REVIEW ({len(review_factors)} 個）：{review_factors}")
print(f"  DROP  ({len(drop_factors)} 個）：{drop_factors}")

print("\n下一步建議：")
print("  1. 在 strategy.py 中移除 DROP 的因子，只保留 KEEP + REVIEW 進入 Fold-Internal IC")
print("  2. 執行 strategy.py 觀察移除不穩定因子後的 CAGR 變化")
print("  3. 接著執行 07_Portfolio_Size_Sweep.ipynb（持股數量掃描）")